# Solving Data Skew with AQE and Broadcast Joins

**What this notebook does**

Joins `transactions` (about 39.8M rows) with `customers` on `cust_id`, where a single customer (`C0YDPQWPBJ`) owns about 17.5M transactions, and times the same join three ways:

1. **Baseline:** sort-merge join with AQE and broadcast disabled; the skew is fully exposed.
2. **AQE skew join:** Spark detects the oversized shuffle partition at runtime and splits it.
3. **Broadcast join:** the small `customers` table is copied to every executor, so there is no shuffle and therefore no skew.

**Core concept: why skew hurts a join**

A sort-merge join shuffles both sides by `hash(cust_id)`, so every row of the hot customer goes to one partition and one task. The stage only finishes when that task finishes, so one oversized partition dictates the runtime while the other cores sit idle.

**Databricks conversion notes**

* Run on an **interactive (all-purpose) cluster** such as *Sandani Personal Compute*. Serverless does not allow changing `spark.sql.adaptive.enabled` or `spark.sql.autoBroadcastJoinThreshold`, so the baseline cannot be reproduced there.
* The local `SparkSession.builder.master("local[4]")` / `sparkContext` setup was removed: Databricks provides `spark`, and `sparkContext` is not available in standard access mode (Spark Connect).
* Local `../../data/data_skew` paths were replaced with the Unity Catalog volume `/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed`.
* Fixed a config typo from the original: `spark.sql.adaptive.skewedJoin.enabled` does not exist; the real setting is `spark.sql.adaptive.skewJoin.enabled`.
* **Timing caveat:** on classic clusters the Databricks disk cache keeps parquet data on local SSD after the first read, so later runs can be faster partly because the data is already cached. Compare runs in the Spark UI (stage and task durations), not only wall-clock time.

## 1. Setup

In [0]:
# Jupyter-only InteractiveShell setting removed; Databricks renders cell output natively.
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')
import time

In [0]:
from pyspark.sql.types import IntegerType
import pyspark.sql.functions as F
# SparkSession is not imported: Databricks provides a ready-to-use `spark` object.

In [0]:
# Original (local Jupyter):
#   spark = SparkSession.builder.master("local[4]").getOrCreate()
#   sc = spark.sparkContext; sc.setLogLevel("ERROR")
# On Databricks the session already exists; sparkContext is not available under Spark Connect.

# Source data in the Unity Catalog volume
base_path = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed"

print(f"Spark version: {spark.version}")
print(f"Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

In [0]:
# spark.conf.set("spark.sql.shuffle.partitions", "3")
# AQE is on by default on Databricks; turn it off so the baseline shows the raw skew.
spark.conf.set("spark.sql.adaptive.enabled", "false")
print("adaptive.enabled =", spark.conf.get("spark.sql.adaptive.enabled"))

## 2. Baseline: Skewed Sort-Merge Join

**What:** Load both parquet datasets, confirm the skew by counting transactions per customer, then run the join with AQE and broadcast joins disabled.

**Core concept:** `countDistinct("txn_id")` per `cust_id` is a quick skew check: one customer with ~17.5M transactions against ~8K for the next ones means its shuffle partition will be thousands of times larger than the average. Setting `autoBroadcastJoinThreshold` to `-1` forces a sort-merge join, so both tables are shuffled and the hot key lands in one task. Transformations (the `join`) are lazy; `count()` is the action that actually runs it.

In [0]:
# Local paths replaced with the Unity Catalog volume
transactions_file = f"{base_path}/transactions.parquet"
customer_file = f"{base_path}/customers.parquet"

df_transactions = spark.read.parquet(transactions_file)
df_customers = spark.read.parquet(customer_file)

In [0]:
df_transactions.printSchema()
display(df_transactions.limit(5))

In [0]:
df_customers.printSchema()
display(df_customers.limit(5))

In [0]:
(
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("txn_id").alias("ct"))
    .orderBy(F.desc("ct"))
    .show(5, False)
)

In [0]:
# -1 disables automatic broadcast joins -> Spark must use a shuffle (sort-merge) join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [0]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [0]:
start_time = time.time()
row_count = df_txn_details.count()
print(f"rows: {row_count:,}")
print(f"time taken: {time.time() - start_time}")

## 3. Fix 1: Adaptive Query Execution (AQE) Skew Join

**What:** Re-enable AQE and its skew-join optimization, then run the same join (broadcast stays disabled).

**Core concept:** AQE re-plans a query at runtime using the actual sizes of shuffle partitions. With `spark.sql.adaptive.skewJoin.enabled`, a partition is treated as skewed when it is larger than both `spark.sql.adaptive.skewJoin.skewedPartitionFactor` (default 5) times the median partition size and `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` (default 256 MB). AQE then **splits the skewed partition into several smaller tasks** and duplicates the matching partition of the other side for each piece, much like automatic salting.

**How to verify:** in the SQL / DataFrame tab of the Spark UI, the `SortMergeJoin` node shows `isSkew=true` and the plan is marked `AdaptiveSparkPlan isFinalPlan=true`. Note that AQE only handles skew in joins, not in aggregations.

In [0]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
# Original used "spark.sql.adaptive.skewedJoin.enabled" (typo, ignored by Spark); correct key below.
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
print("adaptive.enabled          =", spark.conf.get("spark.sql.adaptive.enabled"))
print("adaptive.skewJoin.enabled =", spark.conf.get("spark.sql.adaptive.skewJoin.enabled"))

In [0]:
df_txn_details = (
    df_transactions.join(
        df_customers,
        on="cust_id",
        how="inner"
    )
)

In [0]:
start_time = time.time()
row_count = df_txn_details.count()
print(f"rows: {row_count:,}")
print(f"time taken: {time.time() - start_time}")

## 4. Fix 2: Broadcast Join

**What:** Restore the default 10 MB broadcast threshold and join with an explicit `F.broadcast(df_customers)` hint.

**Core concept:** In a broadcast hash join, the small table is sent once to every executor (classic Spark collects it on the driver first; Databricks can also broadcast executor-to-executor, shown as `Exchange SinglePartition, EXECUTOR_BROADCAST` in the plan), where it is held as an in-memory hash table. Each partition of the large table is joined locally as it is read, so the large side is **never shuffled**. Without a shuffle, rows are not grouped by key, so the hot customer no longer ends up in one task and the skew simply disappears.

**When it applies:** only when one side is small enough to fit in driver and executor memory. `autoBroadcastJoinThreshold` controls automatic broadcasting; the `F.broadcast()` hint forces it regardless of the threshold, so use the hint only for tables you know are small. The plan shows `BroadcastHashJoin` with a broadcast exchange on the customers side only; the transactions side has no shuffle `Exchange`.

In [0]:
# 10MB = 10485760 Bytes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10485760)
print("autoBroadcastJoinThreshold =", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

In [0]:
df_txn_details = (
    df_transactions.join(
        F.broadcast(df_customers),
        on="cust_id",
        how="inner"
    )
)

In [0]:
start_time = time.time()
row_count = df_txn_details.count()
print(f"rows: {row_count:,}")
print(f"time taken: {time.time() - start_time}")

In [0]:
# spark.stop()
# Do not stop the session on Databricks: it is managed by the cluster and shared by the notebook.

## Summary

| Approach | Join strategy | Shuffle of transactions | How skew is handled |
| --- | --- | --- | --- |
| Baseline | Sort-merge | Yes | Not handled: hot `cust_id` is one huge task |
| AQE skew join | Sort-merge, re-planned at runtime | Yes | Skewed partition split into several tasks |
| Broadcast join | Broadcast hash | No | No shuffle, so no key grouping and no skew |

**Choosing a fix:** broadcast when one side is small; rely on AQE (on by default on Databricks) when both sides are large; use manual salting (`4_salting`) when AQE is not enough, or for skewed aggregations.